In [46]:
import numpy as np
import pandas as pd 
import sklearn as sk 
import matplotlib.pyplot as plt
import seaborn as sns

In [47]:
# Reading the training dataset
df = pd.read_csv("Train.csv")

In [48]:
df.columns.tolist()

['Loan_ID',
 'Gender',
 'Married',
 'Dependents',
 'Education',
 'Self_Employed',
 'ApplicantIncome',
 'CoapplicantIncome',
 'LoanAmount',
 'Loan_Amount_Term',
 'Credit_History',
 'Property_Area',
 'Loan_Status']

In [49]:
df["Gender"] = df["Gender"].replace('',np.nan) # Replacing the missing values in the "Gender" column with true NaN values.
df["Gender"] = df.groupby(["Married", "Education", "ApplicantIncome"])["Gender"].transform ( lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Male')) # Filling the NaN values in Gender column based on the grouping of "Married", "Education" and "ApplicantIncome" columns mode. If the mode turns out to be empty, the default value input to the Gender column, replacing the missing values, is to be "Male".

In [50]:
gender_map = {'Female':0, "Male":1} # Creating a map for genders, the values of Gender column wil now be classeified as mentioned (Female as 0, Male as 1)
df['IsMale'] = df["Gender"].map(gender_map) # Creating a new column IsMale so if the person is a male, the value in the column will be 1 otherwise it will be 0. 
df = df.drop(columns= ["Gender"])

In [51]:
df['Has_Coapplicant'] = (df['CoapplicantIncome'] > 0).astype(int) # Create a temporary helper column to check if a co-applicant exists

In [52]:
df["Married"] = df["Married"].replace("", np.nan)
df["Married"] = df.groupby(['Has_Coapplicant', "Dependents"])["Married"].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else "Yes"))

In [53]:
education_map = {'Not Graduate':0, "Graduate":1} # Creating a map for education of the people
df["Is_Graduate"] = df["Education"].map(education_map) # Creating a column for 'Graduate Check' if the person is Graduate then they are mapped to 1 otherwise 0.
df = df.drop(columns=["Education"])

In [54]:
selfEmployment_map = {"Yes":1, "No":0} # Creating a self-employment map
df['Is_Self_Employed'] = df['Self_Employed'].map(selfEmployment_map) # Mapping the self-employed people to 1 and rest to 0.
df = df.drop(columns=['Self_Employed'])

In [55]:
maritalStatus_map = {"Yes":1, "No":0} # Creating a marital-status map
df['Is_Married'] = df['Married'].map(maritalStatus_map)
df = df.drop(columns = ['Married'])

In [58]:
loan_term_mode = df['Loan_Amount_Term'].mode()[0]
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].replace(r" ", np.nan, regex = True)

In [60]:
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(loan_term_mode)

In [63]:
self_employed_mode = df['Is_Self_Employed'].mode()[0]
df['Is_Self_Employed'] = df['Is_Self_Employed'].fillna(self_employed_mode)
df['Is_Self_Employed'] = df['Is_Self_Employed'].astype(int)

In [67]:
df['LoanAmount'] = df['LoanAmount'].fillna(df.groupby(['Is_Graduate', 'Is_Self_Employed'])['LoanAmount'].transform('median'))

In [69]:
df['Dependents'] = df['Dependents'].replace(r"^\s*$", np.nan, regex=True)
df['Dependents'] = df['Dependents'].replace('3+', 3)
dependents_mode = df['Dependents'].mode()[0]
df['Dependents'] = df['Dependents'].fillna(dependents_mode)
df['Dependents'] = df['Dependents'].astype(int)

In [70]:
df['Credit_History'] = df['Credit_History'].replace(" ", np.nan, regex = True)
credit_history_mode = df['Credit_History'].mode()[0]
df['Credit_History'] = df['Credit_History'].fillna(credit_history_mode)
df['Credit_History'] = df['Credit_History'].astype(int)

In [73]:
print(df.columns.tolist())

['Loan_ID', 'Dependents', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Loan_Status', 'IsMale', 'Has_Coapplicant', 'Is_Graduate', 'Is_Self_Employed', 'Is_Married', 'Property_Area_Semiurban', 'Property_Area_Urban']


In [74]:
cols_to_check = ['Loan_ID', 'Dependents', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Loan_Status', 'IsMale', 'Has_Coapplicant', 'Is_Graduate', 'Is_Self_Employed', 'Is_Married', 'Property_Area_Semiurban', 'Property_Area_Urban']
print(df[cols_to_check].isnull().sum())

Loan_ID                     0
Dependents                  0
ApplicantIncome             0
CoapplicantIncome           0
LoanAmount                  0
Loan_Amount_Term            0
Credit_History              0
Loan_Status                 0
IsMale                      3
Has_Coapplicant             0
Is_Graduate                 0
Is_Self_Employed            0
Is_Married                 15
Property_Area_Semiurban     0
Property_Area_Urban         0
dtype: int64


In [75]:
male_mode = df['IsMale'].mode()[0]
df['IsMale'] = df['IsMale'].fillna(male_mode).astype(int)

married_mode = df['Is_Married'].mode()[0]
df['Is_Married'] = df['Is_Married'].fillna(married_mode).astype(int)

In [76]:
print(df[cols_to_check].isnull().sum())

Loan_ID                    0
Dependents                 0
ApplicantIncome            0
CoapplicantIncome          0
LoanAmount                 0
Loan_Amount_Term           0
Credit_History             0
Loan_Status                0
IsMale                     0
Has_Coapplicant            0
Is_Graduate                0
Is_Self_Employed           0
Is_Married                 0
Property_Area_Semiurban    0
Property_Area_Urban        0
dtype: int64


In [78]:
print(df[cols_to_check].dtypes)

Loan_ID                     object
Dependents                   int64
ApplicantIncome              int64
CoapplicantIncome          float64
LoanAmount                 float64
Loan_Amount_Term           float64
Credit_History               int64
Loan_Status                 object
IsMale                       int64
Has_Coapplicant              int64
Is_Graduate                  int64
Is_Self_Employed             int64
Is_Married                   int64
Property_Area_Semiurban      int64
Property_Area_Urban          int64
dtype: object


In [79]:
df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})

df = df.drop(columns=['Loan_ID'])

In [80]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Dependents               614 non-null    int64  
 1   ApplicantIncome          614 non-null    int64  
 2   CoapplicantIncome        614 non-null    float64
 3   LoanAmount               614 non-null    float64
 4   Loan_Amount_Term         614 non-null    float64
 5   Credit_History           614 non-null    int64  
 6   Loan_Status              614 non-null    int64  
 7   IsMale                   614 non-null    int64  
 8   Has_Coapplicant          614 non-null    int64  
 9   Is_Graduate              614 non-null    int64  
 10  Is_Self_Employed         614 non-null    int64  
 11  Is_Married               614 non-null    int64  
 12  Property_Area_Semiurban  614 non-null    int64  
 13  Property_Area_Urban      614 non-null    int64  
dtypes: float64(3), int64(11)
m

In [81]:
X = df.drop(columns = ['Loan_Status'])
y = df['Loan_Status']

In [82]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [84]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
val_predictions = model.predict(X_val)
print(f"Validation Accuracy: {accuracy_score(y_val, val_predictions) * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_val, val_predictions))

Validation Accuracy: 84.86%

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.59      0.71        58
           1       0.84      0.97      0.90       127

    accuracy                           0.85       185
   macro avg       0.87      0.78      0.80       185
weighted avg       0.85      0.85      0.84       185



c:\Users\rashm\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [87]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

continuous_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_val_scaled[continuous_cols] = scaler.transform(X_val[continuous_cols])
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
val_predictions = model.predict(X_val_scaled)

print(f"Scaled Validation Accuracy: {accuracy_score(y_val, val_predictions) * 100:.2f}%")
print("\nUpdated Classification Report:\n", classification_report(y_val, val_predictions))

Scaled Validation Accuracy: 84.86%

Updated Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.59      0.71        58
           1       0.84      0.97      0.90       127

    accuracy                           0.85       185
   macro avg       0.87      0.78      0.80       185
weighted avg       0.85      0.85      0.84       185



In [91]:
test_df = pd.read_csv("Test.csv")
test_df = test_df.replace(r'^\s*$', np.nan, regex=True)

test_df['Is_Married'] = test_df['Married'].map({'Yes': 1, 'No': 0})
test_df['Is_Self_Employed'] = test_df['Self_Employed'].map({'Yes': 1, 'No': 0})
test_df['Is_Graduate'] = test_df['Education'].map({'Graduate': 1, 'Not Graduate': 0})
test_df['IsMale'] = test_df['Gender'].map({'Male': 1, 'Female': 0})
test_df['Dependents'] = test_df['Dependents'].replace('3+', 3)

test_df['Loan_Amount_Term'] = test_df['Loan_Amount_Term'].fillna(loan_term_mode)
test_df['Is_Self_Employed'] = test_df['Is_Self_Employed'].fillna(self_employed_mode)
test_df['Is_Married'] = test_df['Is_Married'].fillna(married_mode)
test_df['IsMale'] = test_df['IsMale'].fillna(male_mode)
test_df['Dependents'] = test_df['Dependents'].fillna(dependents_mode)
test_df['Credit_History'] = test_df['Credit_History'].fillna(credit_history_mode)
test_df['LoanAmount'] =  test_df['LoanAmount'].fillna(df.groupby(['Is_Graduate', 'Is_Self_Employed'])['LoanAmount'].transform('median'))

test_df['Has_Coapplicant'] = (test_df['CoapplicantIncome'] > 0).astype(int)

test_df = pd.get_dummies(test_df, columns=['Property_Area'], drop_first=True, dtype=int)

X_test_final = test_df[X_train.columns].copy()

int_cols = ['Dependents', 'Credit_History', 'IsMale', 'Has_Coapplicant', 'Is_Graduate', 'Is_Self_Employed', 'Is_Married']
X_test_final[int_cols] = X_test_final[int_cols].astype(int)

continuous_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']
X_test_final[continuous_cols] = scaler.transform(X_test_final[continuous_cols])

final_predictions = model.predict(X_test_final)

submission = pd.DataFrame({
    'Loan_ID': test_df['Loan_ID'],
    'Loan_Status': final_predictions
})
submission['Loan_Status'] = submission['Loan_Status'].map({1: 'Y', 0: 'N'})

submission.to_csv('Final Loan Predictions.csv', index=False)
print("Check working directory for 'Final Loan Predictions.csv'.")

Check working directory for 'Final Loan Predictions.csv'.
